# Consumo de un coche según su peso
**Regresión lineal simple · nivel básico** · Proyecto 2 de *Mente Máquina*

Vamos a predecir el **consumo** (litros cada 100 km) de un coche a partir de **un solo dato: su peso**.
Usamos datos reales de 398 coches de los años 1970–1982 (UCI, conjunto nº 9 «Auto MPG», licencia CC BY 4.0).

**Cómo usar este cuaderno:** ejecuta las celdas de arriba abajo, una a una, con el botón ▶ (o `Mayús + Intro`).
Lee antes de cada celda qué hace y por qué. No hace falta saber programar: solo ejecutar y mirar el resultado.

## Preparación: instalar las bibliotecas
Instalamos `ucimlrepo` (para descargar el conjunto de datos) y cargamos las demás herramientas.

In [ ]:
!pip install -q ucimlrepo

import pandas as pd                       # pandas: manejar tablas de datos
import numpy as np                        # numpy: cálculo con números
import matplotlib.pyplot as plt           # matplotlib: dibujar gráficos
from ucimlrepo import fetch_ucirepo      # para descargar el conjunto de datos de UCI
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import statsmodels.api as sm              # statsmodels: estadística de la regresión

print("Todo listo")

## Paso 2 · Descargar y abrir los datos
**Qué hacemos:** descargar la tabla de coches y quedarnos con dos columnas: peso y consumo.
**Cómo:** `fetch_ucirepo(id=9)` pide el conjunto nº 9 de UCI. Los datos vienen en unidades de EE. UU. (peso en libras, consumo en millas por galón), así que los convertimos a kilos y a litros cada 100 km.
**Por qué:** una regresión simple usa una entrada (x) y una salida (y). Aquí x = peso e y = consumo.
**Para qué:** tener los datos en una tabla (`df`) con unidades que entendemos.

In [ ]:
auto = fetch_ucirepo(id=9)                 # descargar el conjunto de datos nº 9 (Auto MPG)

entradas = auto.data.features              # columnas de entrada (peso, cilindrada, potencia...)
salida = auto.data.targets                 # columna de salida (mpg: millas por galón)

# Convertimos a unidades europeas
peso_kg = entradas["weight"] * 0.45359237      # libras -> kilos
consumo = 235.215 / salida["mpg"]              # millas por galón -> litros cada 100 km

df = pd.DataFrame({
    "Peso": peso_kg,                           # kg
    "Consumo": consumo,                        # L/100 km
})

df = df.dropna()                               # quitar filas incompletas, si las hubiera

print("Filas y columnas:", df.shape)

## Paso 3 · Revisar la tabla
**Qué hacemos:** mirar las primeras filas y un resumen de números, y contar los datos que faltan.
**Cómo:** `head()`, `describe()` e `isna().sum()`.
**Por qué:** antes de calcular nada hay que *conocer* los datos: sus unidades, sus rangos y si hay valores vacíos.
**Para qué:** saber entre qué pesos es válido nuestro modelo (lo usaremos en el paso 9).

In [ ]:
print(df.head())                            # las 5 primeras filas
print()
print(df.describe().round(2))              # resumen: media, mínimo, máximo...
print()
print("Valores que faltan en nuestras dos columnas:")
print(df.isna().sum())
print()
print("Valores que faltan en la tabla original (potencia):", int(entradas["horsepower"].isna().sum()), "(no la usamos)")

## Paso 4 · Dibujar el diagrama de dispersión
**Qué hacemos:** un punto por coche: peso en horizontal, consumo en vertical.
**Cómo:** `plt.scatter(x, y)`.
**Por qué:** hay que *mirar antes de calcular*. Si los puntos forman una línea, una recta es buena idea.
**Para qué:** ver la tendencia a ojo. Esperamos una relación **positiva**: más peso, más consumo.

In [ ]:
plt.figure(figsize=(9, 5.5))
plt.scatter(df["Peso"], df["Consumo"], alpha=0.5)
plt.xlabel("Peso del coche (kg)")
plt.ylabel("Consumo (L/100 km)")
plt.title("Peso frente a consumo")
plt.grid(True)
plt.show()

# Un número que resume la relación: el coeficiente de correlación (de -1 a +1)
print("Correlación:", round(df["Peso"].corr(df["Consumo"]), 3))

# ¿Y si hubiéramos usado millas por galón? La relación se curva y la recta encaja peor
print("Correlación con millas por galón:", round(df["Peso"].corr(salida["mpg"]), 3))

## Paso 5 · Separar entrenamiento y prueba
**Qué hacemos:** dividir los coches en dos grupos: 80 % para entrenar y 20 % para examinar.
**Cómo:** `train_test_split(..., test_size=0.20, random_state=42)`.
**Por qué:** si evaluamos con los mismos coches con los que aprendió, es un examen con las preguntas ya vistas.
**Para qué:** medir con honestidad cuánto se equivoca. `random_state=42` hace que el reparto sea siempre el mismo.

In [ ]:
X = df[["Peso"]]                           # entrada (tabla de una columna)
y = df["Consumo"]                          # salida

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,                        # 20 % para prueba
    random_state=42,                       # repartir siempre igual
)

print("Coches de entrenamiento:", len(X_train))
print("Coches de prueba:", len(X_test))

## Paso 6 · Ajustar la recta
**Qué hacemos:** encontrar la mejor recta `consumo = m · peso + b`.
**Cómo:** `LinearRegression().fit(...)` aplica mínimos cuadrados.
**Por qué:** de todas las rectas posibles, la mejor deja el menor error total (SSE).
**Para qué:** obtener la pendiente **m** y el intercepto **b**.

In [ ]:
modelo = LinearRegression()
modelo.fit(X_train, y_train)               # «entrenar»: encontrar la mejor recta con los coches de entrenamiento

pendiente = modelo.coef_[0]                # m: L/100 km que cambian por cada kg
intercepto = modelo.intercept_             # b: consumo estimado a 0 kg (solo un punto de partida matemático)

print(f"Pendiente (m):   {pendiente:.5f} L/100 km por kg")
print(f"Por cada 100 kg más: {pendiente*100:.2f} L/100 km")
print(f"Intercepto (b):  {intercepto:.4f} L/100 km")
signo_b = "−" if intercepto < 0 else "+"
print(f"Ecuación:  Consumo = {pendiente:.5f} · Peso {signo_b} {abs(intercepto):.4f}")

## Paso 7 · Evaluar con coches nuevos
**Qué hacemos:** usar la recta para predecir los coches de prueba y medir el error.
**Cómo:** `predict(X_test)` y tres medidas: **R²**, **RMSE** y **MAE**.
**Por qué:** R² dice cuánto de la variación explica la recta; MAE es el error medio en L/100 km; RMSE castiga más los errores grandes.
**Para qué:** saber si el modelo es fiable. Cuidado: R² **no** es un «porcentaje de aciertos».

In [ ]:
y_pred = modelo.predict(X_test)            # lo que predice la recta para los coches de prueba

comparacion = pd.DataFrame({
    "Peso (kg)": X_test["Peso"].values.round(0),
    "Consumo real": y_test.values.round(2),
    "Consumo predicho": y_pred.round(2),
})
print(comparacion.head(8))
print()

r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print("RESULTADOS DEL MODELO (coches de prueba)")
print(f"R²:    {r2:.4f}")
print(f"RMSE:  {rmse:.3f} L/100 km")
print(f"MAE:   {mae:.3f} L/100 km")

## Paso 8 · Mirar los errores y poner límites
**Qué hacemos:** dibujar la recta sobre los datos y representar los **residuos** (real − predicho).
**Cómo:** `plt.plot` para la recta; `plt.scatter(predicho, residuo)` para los errores.
**Por qué:** un buen R² no basta. Si los errores forman un patrón (una curva, un abanico), la recta se deja algo sin explicar.
**Para qué:** conocer las limitaciones del modelo antes de fiarnos de él.

In [ ]:
# 1) La recta sobre los coches de prueba
x_linea = pd.DataFrame({"Peso": np.linspace(X["Peso"].min(), X["Peso"].max(), 100)})
y_linea = modelo.predict(x_linea)

plt.figure(figsize=(9, 5.5))
plt.scatter(X_test["Peso"], y_test, alpha=0.6, label="Datos reales")
plt.plot(x_linea["Peso"], y_linea, color="red", linewidth=2, label="Recta de regresión")
plt.xlabel("Peso del coche (kg)")
plt.ylabel("Consumo (L/100 km)")
plt.title("Modelo de regresión lineal simple")
plt.legend()
plt.grid(True)
plt.show()

# 2) Los residuos: lo que se le escapa a la recta
residuos = y_test - y_pred
plt.figure(figsize=(9, 4.5))
plt.scatter(y_pred, residuos, alpha=0.6)
plt.axhline(0, color="red", linestyle="--")       # línea de error cero
plt.xlabel("Consumo predicho (L/100 km)")
plt.ylabel("Residuo (L/100 km)")
plt.title("Análisis de residuos")
plt.grid(True)
plt.show()

## Paso 9 · Poner un rango a la predicción
**Qué hacemos:** predecir el consumo de un coche de 1.500 kg y dar un **intervalo**: «entre tanto y tanto, con un 95 % de confianza».
**Cómo:** con `statsmodels`, que además de la recta calcula la incertidumbre.
**Por qué:** una predicción sin margen de error parece más segura de lo que es. Y solo debemos predecir **dentro del rango** de pesos que vimos (paso 3).
**Para qué:** dar una respuesta honesta.

In [ ]:
X_stats = sm.add_constant(X_train)               # statsmodels necesita la columna del intercepto («const»)
modelo_stats = sm.OLS(y_train, X_stats).fit()

nueva = pd.DataFrame({"const": [1.0], "Peso": [1500.0]})            # queremos predecir para 1.500 kg
pred = modelo_stats.get_prediction(nueva).summary_frame(alpha=0.05)  # 95 % de confianza

print("Peso: 1500 kg")
print(f"Consumo estimado: {pred['mean'].iloc[0]:.1f} L/100 km")
print(f"Rango del 95 %:   de {pred['obs_ci_lower'].iloc[0]:.1f} a {pred['obs_ci_upper'].iloc[0]:.1f} L/100 km")
print()
print(modelo_stats.summary().tables[1])
print()
print(f"Rango de pesos con datos: {df['Peso'].min():.0f} a {df['Peso'].max():.0f} kg")
print("¿1.500 kg está dentro del rango?", df['Peso'].min() <= 1500 <= df['Peso'].max())

## Paso 10 · Contar el resultado en cinco frases
**Qué hacemos:** resumir el proyecto con una plantilla de 5 frases: pregunta, datos, resultado, limitaciones y conclusión.
**Cómo:** el cuaderno rellena la plantilla con tus propios números.
**Por qué:** saber contar un resultado es tan importante como calcularlo.
**Para qué:** copiar este texto en tu informe o en la web.

In [ ]:
rango_ini, rango_fin = df["Peso"].min(), df["Peso"].max()
signo = "sube" if pendiente > 0 else "baja"

print("1. Pregunta: ¿cuánto consume un coche según su peso?")
print(f"2. Datos: {len(df)} coches reales de 1970 a 1982, con pesos entre {rango_ini:.0f} y {rango_fin:.0f} kg.")
print(f"3. Resultado: por cada 100 kg más, el consumo {signo} unos {abs(pendiente)*100:.2f} L/100 km (R² = {r2:.2f}: el peso explica el {r2*100:.0f} % de la variación).")
print(f"4. Limitaciones: solo es fiable entre {rango_ini:.0f} y {rango_fin:.0f} kg; son coches antiguos y con una sola variable no explica todo (motor, aerodinámica, año...).")
print(f"5. Conclusión: un coche de 1.500 kg consumiría unos {pred['mean'].iloc[0]:.1f} L/100 km, con un error típico de {mae:.1f} L/100 km (MAE).")

## Extra · Preparar los datos para la web
Esta celda guarda los datos reales y los resultados en un archivo (`datos--consumo-coche-peso.json`) y lo descarga.
Con él, la web dibuja la nube de puntos **real** y la recta que tú has calculado, y deja moverla con deslizadores.

In [ ]:
import json

datos_web = {
    "t": df["Peso"].round(1).tolist(),                       # pesos (kg)
    "p": df["Consumo"].round(2).tolist(),                    # consumos (L/100 km)
    "e": df.index.isin(X_test.index).astype(int).tolist(),   # 1 = dato de prueba, 0 = entrenamiento
    "m": float(pendiente), "b": float(intercepto),           # la recta calculada por el programa
    "r2": float(r2), "rmse": float(rmse), "mae": float(mae),
    "meta": {
        "xl": "Peso del coche (kg)", "yl": "Consumo (L/100 km)", "xu": "kg", "yu": "L/100 km",
        "xn": "Peso", "yn": "Consumo", "xv": "P",
        "md": 4, "bd": 2, "bst": 0.01, "mstep": 0.0001, "mspan": 0.012, "bspan": 3,
        "xd": 0, "yd": 1, "yp": 2,
        "pista": "la pendiente debería ser positiva (más peso, más consumo)",
        "bx": [600, 2400], "by": [0, 30], "ax": [0, 2400], "ay": [-6, 30], "ax1": 1000,
        "tol": [0.5, 4, 0.5, 1.5], "xs": [600, 2400, 50, 1500], "dmax": [50, 500, 50, 200],
        "frase_dentro": "Un coche de {x} {xu} consumiría unos {h} {yu}; el 95 % de las veces el consumo real quedará entre {lo} y {hi} {yu}.",
        "frase_fuera": "⚠ Con {x} {xu} no hay coches reales (solo de {tmin} a {tmax} {xu}). La recta da {h} {yu}, pero es una extrapolación: no te fíes.",
        "frase_sube": "Si el coche pesa {d} {xu} más, el consumo estimado cambia {c} {yu} (pendiente × {d}).",
    },
}

nombre = "datos--consumo-coche-peso.json"
with open(nombre, "w") as fichero:
    json.dump(datos_web, fichero)

from google.colab import files      # solo funciona en Google Colab
files.download(nombre)              # descarga el archivo a tu ordenador

## Retos para consolidar
1. **Básico:** cambia 1.500 kg por 1.000, 1.200, 1.800 y 2.000 kg en el paso 9 y compara las predicciones.
2. **Intermedio:** cambia `test_size` en el paso 5 (0.30, 0.10) y observa cómo cambian R², RMSE y MAE.
3. **Avanzado:** repite el proyecto usando millas por galón (`salida["mpg"]`) como salida y compara R² y residuos: verás por qué conviene medir en litros cada 100 km.

> El peso no es la única causa del consumo: el motor, la aerodinámica o el año del modelo también influyen. La regresión simple muestra una relación, no prueba la causa única.

*Datos: Auto MPG, UCI Machine Learning Repository (CC BY 4.0). Coches de EE. UU., Europa y Japón de 1970–1982. Cuaderno de Mente Máquina · Pedro José de la Rosa Martín.*